In [0]:
from pyspark.sql import Row, functions as F, types as T
from delta.tables import DeltaTable
from datetime import datetime
import requests
import json
import time
import logging
import uuid

In [0]:
# ---------------------------------------------------------------------------
# Logging Configuration
# ---------------------------------------------------------------------------
NOTEBOOK_NAME    = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get().split("/")[-1]

log              = logging.getLogger(NOTEBOOK_NAME)
log.setLevel(logging.INFO)
if not log.handlers:
    _handler     = logging.StreamHandler()
    _handler.setFormatter(
        logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
    )
    log.addHandler(_handler)

# ---------------------------------------------------------------------------
# Dynamic Configuration
# ---------------------------------------------------------------------------
dbutils.widgets.dropdown("env", "dev", ["dev", "tst", "prd"], "Environment")
env         = dbutils.widgets.get("env")
CATALOG          = f"{env}_corpdatav2"
BRONZE_DB        = f"{CATALOG}.bronze"
SILVER_DB        = f"{CATALOG}.silver"
AUDIT_CATALOG    = f"{env}_audit"
LOGS             = f"{AUDIT_CATALOG}.logs"

LOGS_TABLE       = f"{LOGS}.{NOTEBOOK_NAME}"
TARGET_BRONZE    = f"{BRONZE_DB}.AZ_API_ActiveDirectory"
TARGET_SILVER    = f"{SILVER_DB}.AZ_API_ActiveDirectory"

BASE_APIM        = "https://apim.api-dev.seaspancorp.com"
AUTH_APIM        = f"{BASE_APIM}/azad/AuthToken"
GRAPH_API        = "https://graph.microsoft.com/v1.0/users"

APIKEY           = dbutils.secrets.get(scope="azure-scope", key="adb-azad-apim")

AUTH_HEADERS          = {
    "Ocp-Apim-Subscription-Key": f"{APIKEY}",
}

GRAPH_PARAMS     = {"$select": "givenName,surname,displayName,mail,userPrincipalName,signInActivity,createdDateTime,accountEnabled"}
# ---------------------------------------------------------------------------
# Pipeline Settings
# ---------------------------------------------------------------------------
MAX_RETRIES      = 3
BATCH_RUN_ID     = str(uuid.uuid4())

log.info(f"Environment: {env}")
log.info(f"Batch Run ID: {BATCH_RUN_ID}")
log.info(f"Catalog: {CATALOG}")
log.info(f"Max Retries: {MAX_RETRIES}")

In [0]:
audit_schema = T.StructType([
    T.StructField("EventID", T.StringType(), True),
    T.StructField("batch_run_id", T.StringType(), True),
    T.StructField("api_type", T.StringType(), True),
    T.StructField("http_method", T.StringType(), True),
    T.StructField("api_url", T.StringType(), True),
    T.StructField("input_value", T.StringType(), True),
    T.StructField("request_body", T.StringType(), True),
    T.StructField("attempt_number", T.IntegerType(), True),
    T.StructField("request_timestamp", T.TimestampType(), True),
    T.StructField("response_code", T.IntegerType(), True),
    T.StructField("response_body", T.StringType(), True),
    T.StructField("duration_ms", T.LongType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("error_message", T.StringType(), True),
    T.StructField("execution_context", T.StringType(), True),
])

bronze_schema = T.StructType([
    T.StructField("Raw_data", T.StringType(), True),
    T.StructField("Api_metadata", T.StringType(), True),
    T.StructField("_is_normalized", T.BooleanType(), True),
    T.StructField("_Batch_run_id", T.StringType(), True),
    T.StructField("_ingestion_ts", T.TimestampType(), True),
])

sign_in_activity_schema = T.StructType([
    T.StructField("lastSignInDateTime", T.StringType(), True),
    T.StructField("lastSignInRequestId", T.StringType(), True),
    T.StructField("lastNonInteractiveSignInDateTime", T.StringType(), True),
    T.StructField("lastNonInteractiveSignInRequestId", T.StringType(), True),
    T.StructField("lastSuccessfulSignInDateTime", T.StringType(), True),
    T.StructField("lastSuccessfulSignInRequestId", T.StringType(), True),
])

user_record_schema = T.StructType([
    T.StructField("givenName", T.StringType(), True),
    T.StructField("surname", T.StringType(), True),
    T.StructField("displayName", T.StringType(), True),
    T.StructField("mail", T.StringType(), True),
    T.StructField("userPrincipalName", T.StringType(), True),
    T.StructField("createdDateTime", T.StringType(), True),
    T.StructField("accountEnabled", T.BooleanType(), True),
    T.StructField("id", T.StringType(), True),
    T.StructField("signInActivity", sign_in_activity_schema, True),
])

_odata_page_schema = T.StructType([
    T.StructField("@odata.context", T.StringType(), True),
    T.StructField("@odata.nextLink", T.StringType(), True),
    T.StructField("value", T.ArrayType(user_record_schema), True),
])

In [0]:
# ---------------------------------------------------------------------------
# Helper Functions
# ---------------------------------------------------------------------------
def call_api(url, method="POST", headers=None, body=None, data=None, params=None, api_type="sapsf", input_value=None):
    """Pure API caller with retry logic for single-hit endpoints.
    Use `body` for JSON payloads, `data` for form-encoded payloads.
    Returns result dict with all attempts attached.
    Raises RuntimeError on HTTP 400-428 after max retries.
    """
    attempts = []
    last_status_code = None

    for attempt in range(1, MAX_RETRIES + 1):
        event_id = str(uuid.uuid4())
        start_ts = time.time()
        request_ts = datetime.now()

        try:
            response = requests.request(
                method, url, headers=headers, json=body, data=data, params=params, timeout=60
            )
            duration_ms = int((time.time() - start_ts) * 1000)
            last_status_code = response.status_code

            audit_body = json.dumps(body or data) if (body or data) else None

            if 200 <= response.status_code <= 204:
                # Safely parse response — return raw text if not valid JSON
                try:
                    response_data = response.json() if response.text else None
                except (ValueError, json.JSONDecodeError):
                    response_data = response.text

                attempts.append({
                    "EventID": event_id,
                    "batch_run_id": BATCH_RUN_ID,
                    "api_type": api_type,
                    "http_method": method,
                    "api_url": url,
                    "input_value": input_value,
                    "request_body": audit_body,
                    "attempt_number": attempt,
                    "request_timestamp": request_ts,
                    "response_code": response.status_code,
                    "response_body": response.text[:4000],
                    "duration_ms": duration_ms,
                    "status": "SUCCESS",
                    "error_message": None,
                    "execution_context": None,
                })
                return {
                    "success": True,
                    "response_data": response_data,
                    "api_metadata": {
                        "Status": "SUCCESS",
                        "ErrorMessage": None,
                        "HttpStatus": response.status_code,
                        "Timestamp": request_ts.strftime('%Y-%m-%dT%H:%M:%S'),
                    },
                    "attempts": attempts,
                }
            else:
                status = "RETRY" if attempt < MAX_RETRIES else "FAILED"
                attempts.append({
                    "EventID": event_id,
                    "batch_run_id": BATCH_RUN_ID,
                    "api_type": api_type,
                    "http_method": method,
                    "api_url": url,
                    "input_value": input_value,
                    "request_body": audit_body,
                    "attempt_number": attempt,
                    "request_timestamp": request_ts,
                    "response_code": response.status_code,
                    "response_body": response.text[:4000],
                    "duration_ms": duration_ms,
                    "status": status,
                    "error_message": f"HTTP {response.status_code}",
                    "execution_context": None,
                })
                if attempt < MAX_RETRIES:
                    time.sleep(2 ** attempt)

        except requests.exceptions.RequestException as exc:
            duration_ms = int((time.time() - start_ts) * 1000)
            status = "RETRY" if attempt < MAX_RETRIES else "FAILED"
            attempts.append({
                "EventID": event_id,
                "batch_run_id": BATCH_RUN_ID,
                "api_type": api_type,
                "http_method": method,
                "api_url": url,
                "input_value": input_value,
                "request_body": json.dumps(body or data) if (body or data) else None,
                "attempt_number": attempt,
                "request_timestamp": request_ts,
                "response_code": None,
                "response_body": None,
                "duration_ms": duration_ms,
                "status": status,
                "error_message": str(exc),
                "execution_context": None,
            })
            if attempt < MAX_RETRIES:
                time.sleep(2 ** attempt)

    # Max retries exhausted
    if last_status_code and 400 <= last_status_code <= 428:
        raise RuntimeError(
            f"Non-retryable client error (HTTP {last_status_code}) after {MAX_RETRIES} attempts for {url}"
        )

    return {
        "success": False,
        "response_data": None,
        "api_metadata": {
            "Status": "FAILED",
            "ErrorMessage": f"Max retries ({MAX_RETRIES}) exhausted",
            "HttpStatus": last_status_code,
            "Timestamp": datetime.now().strftime('%Y-%m-%dT%H:%M:%S'),
        },
        "attempts": attempts,
    }


def call_paginated_api(base_url, headers, params=None, api_type="azad_users"):
    """Graph API paginated caller. Follows @odata.nextLink across pages.
    Accumulates all value records across pages into a single combined result.
    params is passed only on the first request; @odata.nextLink is self-contained thereafter.
    Returns result dict with all attempts from all pages attached.
    """
    all_records = []
    all_attempts = []
    page_url = base_url
    page_num = 0
    page_params = params  # Only sent on first call; @odata.nextLink carries params thereafter

    while page_url:
        page_num += 1
        log.info(f"  Fetching page {page_num}...")

        page_result = call_api(
            url=page_url,
            method="GET",
            headers=headers,
            params=page_params,
            api_type=api_type,
            input_value=f"page_{page_num}",
        )
        page_params = None  # Clear after first page — @odata.nextLink is self-contained
        all_attempts.extend(page_result["attempts"])

        if not page_result["success"]:
            log.error(f"  Page {page_num} FAILED. Stopping pagination.")
            return {
                "success": False,
                "response_data": all_records if all_records else None,
                "api_metadata": {
                    "Status": "FAILED",
                    "ErrorMessage": f"Pagination failed at page {page_num}",
                    "HttpStatus": page_result["api_metadata"]["HttpStatus"],
                    "Timestamp": datetime.now().strftime('%Y-%m-%dT%H:%M:%S'),
                },
                "attempts": all_attempts,
                "pages_fetched": page_num,
            }

        response_data = page_result["response_data"]
        all_records.append(response_data)
        page_count = len(response_data.get("value", []))
        log.info(f"  Page {page_num}: {page_count} records in response")

        # Follow @odata.nextLink for next page (Graph API pagination)
        page_url = response_data.get("@odata.nextLink", None)

    total_records = sum(len(p.get("value", [])) for p in all_records)
    log.info(f"  Pagination complete. {page_num} pages, {total_records} total records.")
    return {
        "success": True,
        "response_data": all_records,
        "api_metadata": {
            "Status": "SUCCESS",
            "ErrorMessage": None,
            "HttpStatus": 200,
            "Timestamp": datetime.now().strftime('%Y-%m-%dT%H:%M:%S'),
        },
        "attempts": all_attempts,
        "pages_fetched": page_num,
        "total_records": total_records,
    }


def log_audit(result):
    """Write all attempt entries from a result to the audit logs table."""
    exec_context = json.dumps({
        "notebook_path": dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get(),
        "env": env,
    })

    for entry in result["attempts"]:
        entry["execution_context"] = exec_context

    if not result["attempts"]:
        log.warning("No audit entries to write.")
        return

    df = spark.createDataFrame(result["attempts"], schema=audit_schema)
    df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(LOGS_TABLE)
    log.info(f"Wrote {len(result['attempts'])} audit entries to {LOGS_TABLE}")


def write_to_bronze(result):
    """Append one row per API page to bronze. Only writes on SUCCESS with non-empty response.
    Each bronze row contains a single page response as Raw_data JSON string.
    FAILED or empty responses are captured in audit logs only — no bronze write.
    """
    if not (result["success"] and result["response_data"]):
        log.warning("Skipping bronze write: API call failed or returned empty response.")
        return

    pages = result["response_data"]
    rows = [
        Row(
            Raw_data=json.dumps(page),
            Api_metadata=json.dumps(result["api_metadata"]),
            _is_normalized=False,
            _Batch_run_id=BATCH_RUN_ID,
            _ingestion_ts=datetime.now(),
        )
        for page in pages
    ]

    record_count = sum(len(p.get("value", [])) for p in pages if isinstance(p, dict))
    df = spark.createDataFrame(rows, schema=bronze_schema)
    df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(TARGET_BRONZE)
    log.info(f"Appended {len(rows)} rows ({record_count} records) to {TARGET_BRONZE}")

In [0]:
def normalize_to_silver():
    """Read un-normalized bronze rows, parse Raw_data JSON, flatten signInActivity,
    and merge into silver. Marks processed bronze rows as _is_normalized=True.
    """
    all_unnorm_df = spark.read.table(TARGET_BRONZE).filter(F.col("_is_normalized") == False)

    if all_unnorm_df.count() == 0:
        log.info("No un-normalized bronze rows found. Skipping silver write.")
        return

    # Filter to only the latest batch by max _ingestion_ts
    latest_batch_id = (
        all_unnorm_df
        .orderBy(F.col("_ingestion_ts").desc())
        .select("_Batch_run_id")
        .first()[0]
    )
    bronze_df = all_unnorm_df.filter(F.col("_Batch_run_id") == latest_batch_id)
    row_count = bronze_df.count()
    log.info(f"Processing {row_count} un-normalized bronze rows from batch {latest_batch_id}...")

    # Parse Raw_data JSON using page schema
    parsed_df = bronze_df.withColumn("parsed", F.from_json(F.col("Raw_data"), _odata_page_schema))

    # Explode value array to get individual user records
    users_df = parsed_df.select(
        F.explode(F.col("parsed.value")).alias("user"),
        F.col("_Batch_run_id"),
    )

    # Flatten user record and signInActivity struct; cast ISO datetime strings to timestamp
    silver_df = users_df.select(
        F.col("user.givenName").alias("givenName"),
        F.col("user.surname").alias("surname"),
        F.col("user.displayName").alias("displayName"),
        F.col("user.mail").alias("mail"),
        F.col("user.userPrincipalName").alias("userPrincipalName"),
        F.col("user.createdDateTime").alias("createdDateTime"),
        F.col("user.accountEnabled").cast("string").alias("accountEnabled"),
        F.col("user.id").alias("id"),
        F.col("user.signInActivity.lastSignInDateTime").alias("lastSignInDateTime"),
        F.col("user.signInActivity.lastSignInRequestId").alias("lastSignInRequestId"),
        F.col("user.signInActivity.lastNonInteractiveSignInDateTime").alias("lastNonInteractiveSignInDateTime"),
        F.col("user.signInActivity.lastNonInteractiveSignInRequestId").alias("lastNonInteractiveSignInRequestId"),
        F.col("user.signInActivity.lastSuccessfulSignInDateTime").alias("lastSuccessfulSignInDateTime"),
        F.col("user.signInActivity.lastSuccessfulSignInRequestId").alias("lastSuccessfulSignInRequestId"),
        F.current_timestamp().alias("LoadDateUTC"),
    )

    record_count = silver_df.count()
    silver_df.write.format("delta").mode("overwrite").saveAsTable(TARGET_SILVER)
    log.info(f"Wrote {record_count} records to {TARGET_SILVER}")

    # Mark only this batch's bronze rows as normalized
    bronze_table = DeltaTable.forName(spark, TARGET_BRONZE)
    bronze_table.update(
        condition=(F.col("_is_normalized") == False) & (F.col("_Batch_run_id") == latest_batch_id),
        set={"_is_normalized": F.lit(True)},
    )
    log.info(f"Marked bronze rows as normalized for batch {latest_batch_id}.")

In [0]:
# ---------------------------------------------------------------------------
# Main Execution
# ---------------------------------------------------------------------------
if __name__ == "__main__":

    # --- Step 1: Get Azure AD OAuth Token (one-hit) ---
    log.info("Step 1: Requesting Azure AD OAuth token...")
    token_result = call_api(AUTH_APIM, method="POST", headers=AUTH_HEADERS, api_type="azad_token")
    log_audit(token_result)

    if not token_result["success"]:
        raise RuntimeError(f"Token request FAILED: {token_result['api_metadata']['ErrorMessage']}")
    access_token = token_result["response_data"]
    log.info("  OAuth token acquired.")

    # # --- Step 2: Fetch Azure AD Users (Paginated via Graph API) ---
    log.info("Step 2: Fetching Azure AD users (paginated)...")
    GRAPH_HEADERS = {**AUTH_HEADERS, "Authorization": f"Bearer {access_token}"}
    users_result = call_paginated_api(GRAPH_API, headers=GRAPH_HEADERS, params=GRAPH_PARAMS, api_type="azad_users")
    log_audit(users_result)
    write_to_bronze(users_result)
    normalize_to_silver()

    log.info(f"  Pages fetched: {users_result.get('pages_fetched', 0)}, Total records: {users_result.get('total_records', 0)}.")
    log.info(f"  Pipeline completed successfully. Batch Run ID: {BATCH_RUN_ID}")